# 🤝 Ensemble Learning — Kekuatan Tim Lebih Besar dari Individu

**Modul 1: Machine Learning Fundamentals** | Notebook 6 dari 9

---

## 🎯 Apa yang Akan Kita Pelajari?

1. **Voting** — beberapa model berbeda "voting" untuk keputusan akhir
2. **Bagging & Random Forest** — banyak pohon, masing-masing belajar dari data berbeda
3. **Boosting (AdaBoost & Gradient Boosting)** — model belajar dari kesalahan sebelumnya
4. **Stacking** — model bertingkat *(bonus)*

### ⏱️ Estimasi Waktu: ~90 menit + diskusi

---

## 💡 Analogi: Kebijaksanaan Kelompok (Wisdom of Crowds)

Bayangkan ada kuis di kelas:
- Jika **1 siswa** menjawab sendiri → mungkin salah
- Jika **30 siswa** menjawab lalu **diambil suara terbanyak** → kemungkinan benar jauh lebih tinggi!

Inilah prinsip **Ensemble Learning**: **menggabungkan banyak model** yang biasa-biasa saja menjadi satu model yang **sangat kuat**.

### 4 Strategi Ensemble:

| Strategi | Analogi |
|----------|--------|
| **Voting** | 3 juri berbeda menilai lomba, ambil suara terbanyak |
| **Bagging** | 30 siswa masing-masing belajar dari buku berbeda, lalu voting |
| **Boosting** | 1 siswa belajar, temannya fokus memperbaiki soal yang salah |
| **Stacking** | Siswa-siswa menjawab, lalu guru pintar memutuskan jawaban final |

---

## 1️⃣ Persiapan

Kita akan menggunakan dataset sintetis **make_moons** — dua kelompok data berbentuk bulan sabit yang saling bertumpuk. Dataset ini bagus untuk melihat perbedaan antar model.

In [ ]:
# Import library
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Model-model individual
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Ensemble methods
from sklearn.ensemble import (
    VotingClassifier, BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Library berhasil di-import!')

In [ ]:
# Buat dataset make_moons
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

print(f'Data latihan: {len(X_train)} | Data ujian: {len(X_test)}')

# Visualisasi dataset
plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlGn', edgecolors='white', s=40, alpha=0.7)
plt.title('Dataset Make Moons — Dua Bulan Sabit', fontsize=14, fontweight='bold')
plt.xlabel('Fitur 1', fontsize=11)
plt.ylabel('Fitur 2', fontsize=11)
plt.show()

print('💡 Perhatikan: kedua kelas saling bertumpuk — tidak bisa dipisah dengan garis lurus!')

In [ ]:
# Fungsi untuk plot decision boundary (akan dipakai berulang)
def plot_boundary(clf, X, y, ax, title=''):
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                          np.arange(y_min, y_max, 0.02))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlGn', edgecolors='white', s=30, alpha=0.7)
    acc = accuracy_score(y_test, clf.predict(X_test))
    ax.set_title(f'{title}\nAkurasi={acc:.3f}', fontsize=11, fontweight='bold')

print('✅ Fungsi plot_boundary siap digunakan')

---

## 2️⃣ Voting — Beberapa Model Berbeda Melakukan Voting

Ide paling sederhana: latih **beberapa model berbeda**, lalu ambil **suara terbanyak**.

```
  Logistic Regression → prediksi: Kelas A
  Decision Tree       → prediksi: Kelas B
  SVM                 → prediksi: Kelas A
  ─────────────────────────────────────────
  Hasil voting (2 vs 1): Kelas A ✅
```

### Dua jenis voting:
- **Hard Voting** — setiap model memberikan 1 suara (mayoritas menang)
- **Soft Voting** — melihat **probabilitas** dari setiap model, lalu rata-rata (biasanya lebih baik)

In [ ]:
# Latih model-model individual dulu
lr = LogisticRegression(random_state=42)
dt = DecisionTreeClassifier(random_state=42)
svc = SVC(probability=True, random_state=42)

# Voting Classifier (Hard Voting)
voting_hard = VotingClassifier(
    estimators=[('lr', lr), ('dt', dt), ('svc', svc)],
    voting='hard'
)
voting_hard.fit(X_train, y_train)

# Voting Classifier (Soft Voting)
voting_soft = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('dt', DecisionTreeClassifier(random_state=42)),
        ('svc', SVC(probability=True, random_state=42))
    ],
    voting='soft'
)
voting_soft.fit(X_train, y_train)

# Bandingkan individual vs voting
print('📊 Perbandingan: Model Individual vs Voting')
print('=' * 50)
for name, clf in voting_hard.named_estimators_.items():
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f'  {name.upper():15} (individu): {acc:.3f}')
print('-' * 50)
print(f'  {"HARD VOTING":15} (tim):      {voting_hard.score(X_test, y_test):.3f}')
print(f'  {"SOFT VOTING":15} (tim):      {voting_soft.score(X_test, y_test):.3f}')
print('=' * 50)
print('\n💡 Tim (voting) biasanya lebih baik dari individu terkuat!')

---

## 3️⃣ Bagging & Random Forest — Banyak Pohon, Data Berbeda

### Bagging (Bootstrap Aggregating)

Ide: latih **banyak model yang sama** (misalnya Decision Tree), tapi masing-masing belajar dari **sampel data yang berbeda** (diambil acak dengan pengembalian).

**Analogi:** 30 siswa masing-masing hanya membaca **setengah buku** yang berbeda-beda, lalu saat ujian mereka diskusi → jawaban kelompok lebih lengkap dari jawaban individu.

### Random Forest

Random Forest = Bagging + Decision Tree + **fitur acak**. Setiap pohon tidak hanya melihat data berbeda, tapi juga hanya boleh "bertanya" tentang **sebagian fitur** saja. Ini membuat pohon-pohon lebih **beragam** dan hasilnya lebih bagus.

In [ ]:
# Bandingkan: 1 pohon vs Bagging vs Random Forest
tree_single = DecisionTreeClassifier(random_state=42)
tree_single.fit(X_train, y_train)

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    max_samples=100, n_jobs=-1, random_state=42
)
bag_clf.fit(X_train, y_train)

rf_clf = RandomForestClassifier(
    n_estimators=500, max_leaf_nodes=16,
    n_jobs=-1, random_state=42
)
rf_clf.fit(X_train, y_train)

# Visualisasi perbandingan
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, mdl, nama in [
    (axes[0], tree_single, '1 Decision Tree'),
    (axes[1], bag_clf, f'Bagging (500 pohon)'),
    (axes[2], rf_clf, f'Random Forest (500 pohon)')
]:
    plot_boundary(mdl, X_train, y_train, ax, nama)

plt.suptitle('1 Pohon vs Banyak Pohon', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Perhatikan: 1 pohon batasnya "berliku" (overfitting),')
print('   sementara Bagging dan RF batasnya LEBIH HALUS dan akurat!')

### 🌲 Feature Importance — Fitur Mana yang Paling Penting?

Keunggulan Random Forest: bisa menunjukkan **fitur mana yang paling berpengaruh** terhadap prediksi.

In [ ]:
# Contoh feature importance pada dataset Social Network Ads
import pandas as pd

# Gunakan make_moons data yang sudah ada
fi = rf_clf.feature_importances_
fitur_names = ['Fitur 1 (x)', 'Fitur 2 (y)']

plt.figure(figsize=(8, 4))
bars = plt.barh(fitur_names, fi, color=['#2196F3', '#FF6F00'])
plt.xlabel('Importance Score', fontsize=11)
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
for bar, score in zip(bars, fi):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{score:.3f}', va='center', fontsize=12)
plt.tight_layout()
plt.show()

print('💡 Feature Importance menunjukkan seberapa besar kontribusi tiap fitur.')
print('   Ini berguna untuk menentukan fitur mana yang perlu dipertahankan atau dibuang.')

---

## 4️⃣ Boosting — Belajar dari Kesalahan

Berbeda dengan Bagging (semua model dilatih **paralel/bersamaan**), Boosting melatih model secara **berurutan** — setiap model baru **fokus memperbaiki kesalahan** model sebelumnya.

**Analogi:** Bayangkan kamu mengerjakan soal ujian:
1. Percobaan 1: Kamu jawab semua soal → ada 10 yang salah
2. Percobaan 2: Kamu **fokus latihan** pada 10 soal yang salah → tinggal 3 yang salah
3. Percobaan 3: Kamu **fokus** pada 3 soal itu → sekarang semua benar!

### AdaBoost (Adaptive Boosting)

Setiap iterasi, data yang **salah diprediksi** diberi **bobot lebih besar** → model berikutnya lebih memperhatikan data sulit.

In [ ]:
# AdaBoost
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # "Stump" — pohon sangat dangkal
    n_estimators=200,
    learning_rate=0.5,
    algorithm='SAMME',
    random_state=42
)
ada_clf.fit(X_train, y_train)

acc_ada = accuracy_score(y_test, ada_clf.predict(X_test))
print(f'📊 AdaBoost: Akurasi = {acc_ada:.3f}')
print(f'   Jumlah model lemah: {ada_clf.n_estimators}')
print(f'   Setiap model hanya 1 level (stump) — sangat sederhana!')
print(f'   Tapi gabungan 200 stump → model yang kuat!')

# Visualisasi: AdaBoost vs Gradient Boosting vs Random Forest
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, mdl, nama in [
    (axes[0], rf_clf, 'Random Forest\n(Bagging)'),
    (axes[1], ada_clf, 'AdaBoost\n(Boosting)'),
    (axes[2], gb_clf, 'Gradient Boosting\n(Boosting)')
]:
    plot_boundary(mdl, X_train, y_train, ax, nama)

plt.suptitle('Bagging vs Boosting — Decision Boundary', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Tidak ada model yang SELALU terbaik! Hasilnya tergantung dataset:')
print('   - Bagging (RF) lebih tahan terhadap NOISE dalam data')
print('   - Boosting lebih kuat jika data bersih dan pola-nya kompleks')
print('   - Selalu coba beberapa metode dan bandingkan hasilnya!')

In [ ]:
# Gradient Boosting
gb_clf = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_clf.fit(X_train, y_train)

acc_gb = accuracy_score(y_test, gb_clf.predict(X_test))
print(f'📊 Gradient Boosting: Akurasi = {acc_gb:.3f}')

In [ ]:
# Visualisasi: AdaBoost vs Gradient Boosting vs Random Forest
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, mdl, nama in [
    (axes[0], rf_clf, 'Random Forest\n(Bagging)'),
    (axes[1], ada_clf, 'AdaBoost\n(Boosting)'),
    (axes[2], gb_clf, 'Gradient Boosting\n(Boosting)')
]:
    plot_boundary(mdl, X_train, y_train, ax, nama)

plt.suptitle('Bagging vs Boosting — Decision Boundary', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Boosting biasanya lebih akurat, tapi lebih lambat karena model dilatih berurutan.')
print('   Bagging (RF) lebih cepat karena model dilatih bersamaan (paralel).')

---

## 🌟 BONUS: Stacking — Model Bertingkat

**Stacking** menggabungkan prediksi beberapa model, lalu **melatih model baru** untuk memutuskan jawaban final.

```
Layer 1 (Model Dasar):
  Logistic Regression → prediksi
  Random Forest       → prediksi  → Gabungkan → Layer 2 (Meta Model):
  SVM                 → prediksi               Random Forest → Jawaban Final
```

**Analogi:** Seperti lomba debat:
- 3 tim masing-masing memberi argumen (model dasar)
- 1 juri ahli mempertimbangkan semua argumen dan membuat keputusan final (meta model)

In [ ]:
# Stacking
stacking_clf = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('svc', SVC(probability=True, random_state=42))
    ],
    final_estimator=RandomForestClassifier(n_estimators=100, random_state=43),
    cv=5
)
stacking_clf.fit(X_train, y_train)

acc_stack = accuracy_score(y_test, stacking_clf.predict(X_test))
print(f'📊 Stacking: Akurasi = {acc_stack:.3f}')

---

## 5️⃣ Perbandingan Besar: Semua Model Ensemble

In [ ]:
# Kumpulkan semua model
all_models = {
    '1 Decision Tree': tree_single,
    'Voting (Soft)': voting_soft,
    'Bagging (500 pohon)': bag_clf,
    'Random Forest': rf_clf,
    'AdaBoost': ada_clf,
    'Gradient Boosting': gb_clf,
    'Stacking': stacking_clf,
}

# Tabel perbandingan
print('📊 PERBANDINGAN SEMUA MODEL ENSEMBLE')
print('=' * 55)
print(f'{"Model":<25} {"Accuracy":>10} {"F1-Score":>10}')
print('-' * 55)

results = []
for nama, mdl in all_models.items():
    y_pred = mdl.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append((nama, acc, f1))
    marker = ' 🏆' if acc == max(r[1] for r in results) else ''
    print(f'  {nama:<23} {acc:>10.3f} {f1:>10.3f}{marker}')

print('=' * 55)

# Bar chart
names = [r[0] for r in results]
accs = [r[1] for r in results]
colors = ['#EF5350'] + ['#42A5F5'] * 6  # Merah untuk single tree, biru untuk ensemble
colors[accs.index(max(accs))] = '#66BB6A'  # Hijau untuk terbaik

plt.figure(figsize=(12, 5))
bars = plt.barh(names, accs, color=colors)
plt.xlabel('Akurasi', fontsize=12)
plt.title('Perbandingan Akurasi Semua Model', fontsize=14, fontweight='bold')
plt.xlim(0.8, 1.0)

for bar, acc in zip(bars, accs):
    plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{acc:.3f}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

print('🔴 Merah = model individual | 🔵 Biru = ensemble | 🟢 Hijau = terbaik')

---

## 📝 Ringkasan: Apa yang Kita Pelajari Hari Ini?

### Konsep Utama

1. **Ensemble = menggabungkan banyak model** untuk hasil yang lebih baik
2. **Voting** = beberapa model berbeda, ambil suara terbanyak
3. **Bagging (Random Forest)** = banyak pohon, masing-masing belajar dari data acak berbeda → mencegah overfitting
4. **Boosting (AdaBoost, Gradient Boosting)** = model berurutan, yang baru memperbaiki kesalahan yang lama → meningkatkan akurasi
5. **Stacking** = model bertingkat dengan meta-learner

### Kapan Menggunakan Apa?

| Metode | Kapan Digunakan | Kecepatan |
|--------|----------------|----------|
| **Random Forest** | Default terbaik, hampir selalu bagus | Cepat (paralel) |
| **Gradient Boosting** | Butuh akurasi maksimal, data tabular | Sedang |
| **Voting** | Sudah punya beberapa model, ingin digabung | Tergantung model |
| **Stacking** | Kompetisi ML, butuh performa terbaik | Lambat |

### Tips Praktis

- 🌲 **Random Forest** = pilihan aman untuk hampir semua masalah
- 🚀 **Gradient Boosting** = pilihan untuk performa terbaik (tapi perlu tuning)
- 🏆 Di kompetisi Kaggle, **Gradient Boosting** (terutama XGBoost/LightGBM) hampir selalu menang!

---

### 🔜 Notebook Selanjutnya: XGBoost
Kita akan belajar **XGBoost** — versi "super" dari Gradient Boosting yang menjadi senjata rahasia pemenang kompetisi Machine Learning!